<a href="https://colab.research.google.com/github/rifal000/Book_RAG_Assistant/blob/main/Book_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

upload document and install dependencies

In [1]:
!pip install -q langchain langchain-community langchain-google-genai faiss-cpu pypdf
!pip install -q langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.3/73.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
from google.colab import files
uploaded = files.upload()

Saving So Late.pdf to So Late.pdf


Load and chunk the PDF

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

/tmp/ipykernel_1369/2300941456.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [6]:
PDF_PATH = "So Late.pdf"

In [7]:
#Create the loader
loader = PyPDFLoader(PDF_PATH)
documents = loader. load()
print(f"Loaded {len(documents)} pages from PDF")

Loaded 34 pages from PDF


In [8]:
#Chunk document
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " "]
)
chunks = splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks")

Split into 69 chunks


Create the embedding and FAISS index

In [9]:
import os
from google.colab import userdata

In [10]:
#Set the OpenAI key using ColabService
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

In [11]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

In [13]:
from re import search
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.25, "k": 12}
)


Create QA chain

In [14]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [15]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
)

In [16]:
prompt = ChatPromptTemplate.from_template("""
You are a Book RAG Assistant.

Your job is to answer questions about the book using ONLY the information provided in the context.

Rules:
1. Do not use outside knowledge.
2. Do not invent characters, events, places, or details.
3. If the answer cannot be found in the context, say: "Not found in the book."
4. For questions about characters, describe them only based on the book.
5. For questions about events, explain what happened based on the book.
6. For questions about relationships, explain the relationship using information from the book.
7. If asked to summarize, summarize only the provided content.
8. When possible, mention the page number where the information appears.
9. If the question refers to something that happened earlier or later in the story, use the retrieved context to answer if possible.
10. Keep answers clear and concise.

Context:
{context}

Question:
{question}

Answer:
""")

In [17]:
rag_chain = (
{
"context": retriever,
"question": lambda x: x
}
| prompt
| llm
| StrOutputParser()
)

testing the model

In [22]:
rag_chain.invoke("who is the main character")

'Based on the provided context, the main character is **Cathal** (pages 8, 14, 22). The story follows his experiences, interactions, and thoughts.'

Create the Gradio API

In [19]:
!pip install -q gradio

In [24]:
import gradio as gr


def chat_with_book(message, history):
    if not message.strip():
        return

    result = rag_chain.invoke(message)

    yield result


demo = gr.ChatInterface(
    fn=chat_with_book,
    title="Book RAG Assistant",
    description="Ask questions about the book and get answers based only on its content.",
    textbox=gr.Textbox(
        placeholder="Ask something about the book..."
    ),
    examples=[
        "Who is the main character?",
        "Who are the important characters?",
        "What happened at the beginning of the story?",
        "Summarize the story."
    ]
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5d58a49db5bc2502e4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
